In [4]:
import pandas as pd
import dask.dataframe as dd
import openpyxl as px
import os as os
import glob as glob
import csv as csv
import openpyxl as px

In [ ]:
# Load the CSV file
df = dd.read_csv('despesas-2025.csv',
                 encoding='latin1',
                 on_bad_lines='skip',
                 sep=';',
                 assume_missing=True)

# View the first 5 rows
print(df.head())
print(len(df))


   id_despesa_detalhe  ano_exercicio  ds_municipio  codigo_municipio_ibge  \
0         635469579.0         2025.0  Bady Bassitt              3504602.0   
1         635469581.0         2025.0  Bady Bassitt              3504602.0   
2         635469624.0         2025.0  Bady Bassitt              3504602.0   
3         635469627.0         2025.0  Bady Bassitt              3504602.0   
4         635469739.0         2025.0  Bady Bassitt              3504602.0   

                           ds_orgao  mes_referencia mes_ref_extenso  \
0  CÂMARA MUNICIPAL DE BADY BASSITT             1.0         Janeiro   
1  CÂMARA MUNICIPAL DE BADY BASSITT             1.0         Janeiro   
2  CÂMARA MUNICIPAL DE BADY BASSITT             1.0         Janeiro   
3  CÂMARA MUNICIPAL DE BADY BASSITT             1.0         Janeiro   
4  CÂMARA MUNICIPAL DE BADY BASSITT             1.0         Janeiro   

  tp_despesa nr_empenho tp_identificador_despesa  ...  ds_subfuncao_governo  \
0  Empenhado    22-2025   CNPJ 

A seguir devemos transformar as tabelas relacionadas ao PIB de .xlsx para .csv, mantendo a formatação de maneira idônea:

In [ ]:
def limpar_dado(valor):
    if valor is None:
        return ""
        
    # 1. O openpyxl leu como Número (Float ou Int)
    # O Python converte internamente para ponto (Ex: 152363.552). 
    # Nós forçamos a volta para vírgula.
    if isinstance(valor, (int, float)):
        return str(valor).replace('.', ',')
        
    # 2. O openpyxl leu como Texto (String)
    # Ex: "1.127.093.826" ou "152363,552"
    if isinstance(valor, str):
        texto = valor.strip()
        
        # Verifica se é apenas um valor numérico mascarado de texto
        if texto.replace('.', '').replace(',', '').replace('-', '').isdigit():
            # Como a vírgula já é o decimal, basta apagar os pontos de milhar.
            # O Excel vai ler "246.679.409" como "246679409" e interpretar perfeitamente.
            return texto.replace('.', '')

        return texto # Retorna textos normais (Ex: "Águas de São Pedro") sem alterar

    return valor

# --- Execução Principal ---
arquivos_excel = glob.glob("tab_pib_*.xlsx")

if not arquivos_excel:
    print("Nenhum arquivo .xlsx encontrado na pasta.")
else:
    print(f"Encontrados {len(arquivos_excel)} arquivos. Iniciando conversão...\n")

    for caminho_arquivo in arquivos_excel:
        nome_base = os.path.splitext(caminho_arquivo)[0]
        nome_arquivo_csv = f"{nome_base}.csv"
        
        print(f"Convertendo: {caminho_arquivo} -> {nome_arquivo_csv}")

        # Abre o Excel pegando apenas os valores (ignorando fórmulas)
        workbook = px.load_workbook(caminho_arquivo, data_only=True)
        planilha = workbook.active

        # O delimiter=';' garante que as colunas fiquem separadas corretamente no Brasil
        with open(nome_arquivo_csv, mode='w', newline='', encoding='utf-8') as arquivo_saida:
            escritor_csv = csv.writer(arquivo_saida, delimiter=';')
            
            for linha in planilha.iter_rows(values_only=True):
                # Limpa cada célula da linha antes de gravar
                linha_limpa = [limpar_dado(celula) for celula in linha]
                escritor_csv.writerow(linha_limpa)
        
        workbook.close()

    print("\nProcesso finalizado com sucesso!")

Encontrados 10 arquivos. Iniciando conversão...

Convertendo: tab_pib_2009.xlsx -> tab_pib_2009.csv
Convertendo: tab_pib_2010.xlsx -> tab_pib_2010.csv
Convertendo: tab_pib_2011.xlsx -> tab_pib_2011.csv
Convertendo: tab_pib_2012.xlsx -> tab_pib_2012.csv
Convertendo: tab_pib_2013.xlsx -> tab_pib_2013.csv
Convertendo: tab_pib_2014.xlsx -> tab_pib_2014.csv
Convertendo: tab_pib_2015.xlsx -> tab_pib_2015.csv
Convertendo: tab_pib_2016-1.xlsx -> tab_pib_2016-1.csv
Convertendo: tab_pib_2017.xlsx -> tab_pib_2017.csv
Convertendo: tab_pib_2018.xlsx -> tab_pib_2018.csv

Processo finalizado com sucesso!
